# Nova OS v0.77 — Boot-to-Home Integration Preview
This is a **browser preview and source test**, not an ARM64 QEMU boot or flashable phone image.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os, subprocess
uploads=files.upload()
archive=Path(next(k for k in uploads if k.endswith('.zip')))
dest=Path('/content/nova-v0.77')
dest.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archive) as z:
    bad=z.testzip()
    if bad: raise ValueError('Corrupt ZIP member: '+bad)
    z.extractall(dest)
roots=list(dest.glob('*/graphics-shell/server.py'))
assert len(roots)==1, 'The Nova v0.77 ZIP has an unexpected structure'
root=roots[0].parents[1]
print('Nova source:',root)


In [ ]:
import sys
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-q'],cwd=root,check=True)
print('Source tests complete. This does not confirm ARM64 boot.')


In [ ]:
import socket, time, urllib.request
port=8797
try:
    urllib.request.urlopen(f'http://127.0.0.1:{port}/api/health',timeout=1)
    raise RuntimeError('Port already in use; stop the earlier Nova server.')
except urllib.error.URLError: pass
env={**os.environ,'NOVA_COLAB_MODE':'1','NOVA_ENFORCE_AUTH':'1',
     'NOVA_CONFIG_DIR':'/content/nova-v077-user-config','NOVA_FILES_ROOT':'/content/nova-v077-user-files'}
server=subprocess.Popen([sys.executable,str(root/'graphics-shell/server.py'),
    '--no-browser','--host','127.0.0.1','--port',str(port)],env=env,cwd=root)
for _ in range(80):
    try:
        h=urllib.request.urlopen(f'http://127.0.0.1:{port}/api/health',timeout=.5)
        if h.status==200: break
    except Exception: pass
    time.sleep(.25)
else: raise RuntimeError('Nova failed to start; check previous output')
print('Nova browser preview listening at',port)


In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(port)
